# Hands-On Data Lab Implementation — Titanic Dataset
**Data Science Consultant Internship — Task 2**

This notebook walks through a complete, practical data science workflow on the Titanic dataset: environment setup, data import & exploration, data manipulation, data cleaning, visualization, and statistical analysis with insights.

## 1. Set Up the Data Science Environment
Import Python, Pandas, NumPy, and the visualization libraries Matplotlib and Seaborn, and configure default plot styling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

print("Environment ready")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Seaborn:", sns.__version__)

## 2. Import & Explore the Data
Load the Titanic dataset directly from Seaborn's built-in dataset repository (no manual download needed), then examine its structure, column data types, and summary statistics.

In [ ]:
# Load the Titanic dataset (built into Seaborn)
df = sns.load_dataset('titanic')

# Basic structure
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
# Column data types and non-null counts
df.info()

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

In [ ]:
# Check missing values in every column
df.isnull().sum().sort_values(ascending=False)

**Observations:**
- The dataset has 891 passenger records with 15 columns covering demographics (age, sex), travel class, fare, embarkation port, and the survival outcome.
- `deck` has the most missing values, followed by `age` and `embark_town`/`embarked`.
- `survived` (0/1) is the target variable for any downstream classification task.

## 3. Data Manipulation
Practice filtering, sorting, grouping, merging-style transformations, and creating new columns from existing ones using Pandas.

In [ ]:
# Filtering: female passengers travelling in 1st class
first_class_women = df[(df['sex'] == 'female') & (df['pclass'] == 1)]
print("First-class women passengers:", len(first_class_women))
first_class_women[['sex', 'pclass', 'age', 'fare', 'survived']].head()

In [ ]:
# Sorting: passengers sorted by fare paid, highest first
df.sort_values(by='fare', ascending=False)[['sex', 'pclass', 'fare', 'survived']].head(10)

In [ ]:
# Grouping: survival rate by passenger class and gender
survival_by_class_gender = df.groupby(['pclass', 'sex'])['survived'].mean().round(3)
survival_by_class_gender

In [ ]:
# Grouping: average fare and age by embarkation town
df.groupby('embark_town')[['fare', 'age']].mean().round(2)

In [ ]:
# Column transformation: create an 'age_group' column from 'age'
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

# Column transformation: family size from sibsp + parch
df['family_size'] = df['sibsp'] + df['parch'] + 1

df[['age', 'age_group', 'sibsp', 'parch', 'family_size']].head()

## 4. Data Cleaning
Handle missing values, duplicate records, and incorrect data types to prepare the dataset for reliable analysis.

In [ ]:
# Impute missing 'age' values with the median age
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
print(f"Missing age values filled with median: {median_age}")

In [ ]:
# 'deck' has too many missing values to impute meaningfully — drop it
df = df.drop(columns=['deck'])

# Fill the few missing 'embarked'/'embark_town' values with the most frequent port (mode)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Check for and remove duplicate rows
duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

In [ ]:
# Correct data types: convert categorical text columns to category dtype for efficiency
cat_cols = ['sex', 'embarked', 'embark_town', 'class', 'who', 'alive']
for col in cat_cols:
    df[col] = df[col].astype('category')

df.dtypes

## 5. Data Visualization
Create Seaborn/Matplotlib charts to visualize survival patterns, distributions, and correlations.

In [ ]:
# Bar chart: survival rate by gender
plt.figure(figsize=(6, 5))
sns.barplot(data=df, x='sex', y='survived', palette='Set2', errorbar=None)
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Rate')
plt.xlabel('Gender')
plt.show()

In [ ]:
# Histogram: age distribution
plt.figure(figsize=(7, 5))
sns.histplot(df['age'], bins=25, kde=True, color='steelblue')
plt.title('Age Distribution of Passengers')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
# Box plot: fare distribution by passenger class
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x='pclass', y='fare', palette='Set3')
plt.title('Fare Distribution by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Fare')
plt.show()

In [ ]:
# Heatmap: correlation between numeric features
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

## 6. Basic Data Analysis & Insights
Summary statistics and key takeaways from the analysis above.

In [ ]:
# Key summary statistics
overall_survival_rate = df['survived'].mean()
print(f"Overall survival rate: {overall_survival_rate:.2%}")

print("\nSurvival rate by class:")
print(df.groupby('pclass')['survived'].mean().round(3))

print("\nSurvival rate by gender:")
print(df.groupby('sex')['survived'].mean().round(3))

print("\nAverage fare by class:")
print(df.groupby('pclass')['fare'].mean().round(2))

### Key Insights

- **Gender was the strongest predictor of survival.** Women survived at a much higher rate than men, reflecting the "women and children first" evacuation protocol.
- **Passenger class strongly influenced survival.** 1st-class passengers had the highest survival rate, 3rd-class the lowest — likely linked to cabin location and proximity to lifeboats.
- **Fare and class are closely related.** 1st-class passengers paid substantially higher fares on average, and fare itself positively correlates with survival.
- **Age distribution** is concentrated between 20–40 years, with a secondary smaller peak for infants/young children, who also had a relatively higher survival rate in several class/gender groups.
- **Family size** (via engineered `family_size` column) shows passengers travelling with a small family (2–4 members) tended to survive more than those travelling completely alone or in very large families.

**Conclusion:** Socio-economic status (class/fare) and gender were the two dominant factors influencing survival aboard the Titanic, consistent with historical accounts of the evacuation priorities.